# SustainaVERSE Cloud exploration

In [4]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import datasets
import math
import json
from ortools.linear_solver import pywraplp

In [56]:
npg_df = gpd.read_parquet(
    "/Users/vlad/data/smart-meter.parquet",
    filters=[
        ("dno_alias", "==", "NPg"),
    ],
)

npg_df

,dataset_id,dno_alias,aggregated_device_count_active,total_consumption_active_import,data_collection_log_timestamp,geometry,secondary_substation_unique_id,lv_feeder_unique_id
0,NPg_Feeder_February2024,NPg,11,1103,2024-02-01 00:00:00+00:00,POINT (-1.76766 53.84106),127103300-ALBERT STREET 48245,7292694-103-7292694-103
1,NPg_Feeder_February2024,NPg,22,1863,2024-02-01 00:00:00+00:00,POINT (-1.42857 55.0222),148549125-ABBEY,57818-A-57818-A
2,NPg_Feeder_February2024,NPg,27,2162,2024-02-01 00:00:00+00:00,POINT (-1.42857 55.0222),148549125-ABBEY,57818-B-57818-B
3,NPg_Feeder_February2024,NPg,18,2144,2024-02-01 00:00:00+00:00,POINT (-1.42857 55.0222),148549125-ABBEY,57818-D-57818-D
4,NPg_Feeder_February2024,NPg,14,1304,2024-02-01 00:00:00+00:00,POINT (-1.51861 55.01546),148549127-AGRICOLA,57820-A-57820-A
...,...,...,...,...,...,...,...,...
5040651,NPg_Feeder_February2024,NPg,5,242,2024-02-29 23:30:00+00:00,POINT (-1.6726 55.05138),172372061-DONKEY FIELD,7748526-E-7748526-E
5040652,NPg_Feeder_February2024,NPg,5,182,2024-02-29 23:30:00+00:00,POINT (-1.2294 54.68876),172438824-HARTLEPOOL ZETLAND,7765386-C-7765386-C
5040653,NPg_Feeder_February2024,NPg,6,1008,2024-02-29 23:30:00+00:00,POINT (-1.56607 54.53453),174402375-HOLLYHURST NORTH,7802106-D-7802106-D
5040654,NPg_Feeder_February2024,NPg,5,214,2024-02-29 23:30:00+00:00,POINT (-1.56607 54.53453),174402375-HOLLYHURST NORTH,7802106-E-7802106-E


In [57]:
npg_df.explore()

KeyboardInterrupt: 

In [ ]:
faraday_dataset = datasets.load_dataset("OpenSynth/cnz-faraday-4.0", split="train")
faraday_df = pd.DataFrame(faraday_dataset)
faraday_df['kwh_list'] = faraday_df['kwh'].apply(lambda x: json.loads(x.replace("'", '"')))

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

In [7]:
ukpn_substations_df = gpd.read_parquet(
    "ukpn-secondary-sites.parquet"
)
ukpn_substations_df["kva_per_customer"] = (ukpn_substations_df["onanrating"] / ukpn_substations_df["customer_count"])
ukpn_substations_df["kva_per_customer_rounded"] = ukpn_substations_df["kva_per_customer"].round().astype(int)

npg_substations_df = gpd.read_parquet(
    "npg-substations.parquet"
)

In [61]:
# Get all data for the Seventh Street Horden substation
horden_data = npg_df.loc[npg_df['secondary_substation_unique_id'] == '148553006-SEVENTH STREET HORDEN'].copy()
horden_data = horden_data.sort_values('data_collection_log_timestamp')

# Group horden data by feeder and timestamp
grouped_horden = horden_data.groupby(['lv_feeder_unique_id', 'data_collection_log_timestamp'])

# Get all timestamps
horden_timestamps = horden_data['data_collection_log_timestamp'].unique()

# Get unique feeder IDs
feeder_ids = horden_data['lv_feeder_unique_id'].unique()

In [62]:
kva_rating = npg_substations_df.loc[npg_substations_df["site_name"] == "SEVENTH STREET HORDEN"]["kva_rating"].values[0]

# Get the empirical distribution data
kva_per_customer_values = ukpn_substations_df["kva_per_customer"].dropna().values

# Drop zeros from kva_per_customer_values to avoid division by zero
kva_per_customer_values = kva_per_customer_values[kva_per_customer_values != 0]

# Perform Monte Carlo simulation
n_simulations = 100000  # Number of simulations to run
simulated_customer_count = []

for _ in range(n_simulations):
    # Randomly sample from the empirical distribution
    sampled_kva_per_customer = np.random.choice(kva_per_customer_values)
    # Add a bit of noise to counteract the rounding in kva_per_customer
    sampled_kva_per_customer = sampled_kva_per_customer * (1 + np.random.normal(0, 0.07))
    # Calculate estimated number of customers
    estimated_customers = kva_rating / sampled_kva_per_customer
    simulated_customer_count.append(round(estimated_customers))

# Convert to numpy array for easier analysis
simulated_customer_count = np.array(simulated_customer_count)

# Analyze the results
print(f"Mean estimated customers: {np.mean(simulated_customer_count):.1f}")
print(f"Median estimated customers: {np.median(simulated_customer_count):.1f}")
print(f"95% confidence interval: [{np.percentile(simulated_customer_count, 2.5):.1f}, {np.percentile(simulated_customer_count, 97.5):.1f}]")
upper_bound = np.percentile(simulated_customer_count, 97.5)

# Visualize the distribution
fig = go.Figure(data=[go.Histogram(
    x=simulated_customer_count,
    xbins=dict(start=0, end=upper_bound + 1, size=1),
    name='Simulated Number of Customers'
)])

fig.update_layout(
    title='Distribution of Simulated Number of Customers',
    xaxis_title='Number of Customers',
    yaxis_title='Simulations Count'
)

fig.show()

Mean estimated customers: 137.0
Median estimated customers: 118.0
95% confidence interval: [8.0, 379.0]


In [ ]:
def optimize_topology(n_customers, feeder_ids, horden_timestamps, grouped_horden, faraday_df):
    # Create the solver
    solver = pywraplp.Solver.CreateSolver('SAT')
    if not solver:
        return None

    # Create binary variables for customer assignments
    # x[i,j] = 1 if customer i is assigned to feeder j
    x = {}
    for i in range(len(faraday_df)):
        for j in range(len(feeder_ids)):
            x[i,j] = solver.BoolVar(f'x_{i}_{j}')

    # Constraint: Each customer must be assigned to exactly one feeder
    for i in range(len(faraday_df)):
        solver.Add(sum(x[i,j] for j in range(len(feeder_ids))) <= 1)

    # Constraint: Total number of assigned customers must equal n_customers
    total_customers = solver.RowConstraint(float(n_customers), float(n_customers))
    for i in range(len(faraday_df)):
        for j in range(len(feeder_ids)):
            total_customers.SetCoefficient(x[i,j], 1)

    # Create variables for the squared errors
    errors = {}
    for t_idx, timestamp in enumerate(horden_timestamps):
        for j, feeder_id in enumerate(feeder_ids):
            # Get actual measurement
            actual = grouped_horden.get_group(
                (feeder_id, timestamp))['total_consumption_active_import'].iloc[0]
            
            # Get half-hour index
            half_hour_idx = timestamp.hour * 2 + (timestamp.minute // 30)
            
            # Create variable for simulated consumption
            sim_consumption = solver.NumVar(0, solver.infinity(), f'sim_{t_idx}_{j}')
            
            # Add constraint linking simulation to customer assignments
            solver.Add(sim_consumption == sum(
                x[i,j] * faraday_df['kwh_list'].iloc[i][half_hour_idx]
                for i in range(len(faraday_df))
            ))
            
            # Create variable for squared error
            error = solver.NumVar(0, solver.infinity(), f'error_{t_idx}_{j}')
            errors[t_idx,j] = error
            
            # Add constraints to model squared error
            # Since we can't directly square variables, we'll use absolute value
            solver.Add(error >= sim_consumption - actual)
            solver.Add(error >= actual - sim_consumption)

    # Minimize total error
    solver.Minimize(sum(errors[t,j] for t in range(len(horden_timestamps)) 
                                   for j in range(len(feeder_ids))))

    # Solve
    status = solver.Solve()
    
    # Print solver status
    if status == pywraplp.Solver.OPTIMAL:
        print("Optimal solution found!")
    elif status == pywraplp.Solver.FEASIBLE:
        print("Feasible solution found (not necessarily optimal)")
    elif status == pywraplp.Solver.ABNORMAL:
        print("Solver abnormal")
        return None, None
    elif status == pywraplp.Solver.UNBOUNDED:
        print("Solver unbounded")
        return None, None
    elif status == pywraplp.Solver.MODEL_INVALID:
        print("Solver model invalid")
        return None, None
    elif status == pywraplp.Solver.NOT_SOLVED:
        print("Solver didn't solve")
        return None, None
    elif status == pywraplp.Solver.INFEASIBLE:
        print("Solver infeasible")
        return None, None
    else:
        print(f"Solver failed with status: {status}")
        return None, None
    
    # Extract solution regardless of optimality
    customer_indices_by_feeder = [[] for _ in feeder_ids]
    total_assigned = 0
    
    for i in range(len(faraday_df)):
        for j in range(len(feeder_ids)):
            if x[i,j].solution_value() > 0.5:  # Using 0.5 to handle floating-point
                customer_indices_by_feeder[j].append(i)
                total_assigned += 1
    
    print(f"Total customers assigned: {total_assigned} (target: {n_customers})")
    
    return customer_indices_by_feeder, solver.Objective().Value()

# Use the optimizer instead of random search
results = []
sample_size = 200
sampled_timestamps = np.random.choice(horden_timestamps, size=min(sample_size, len(horden_timestamps)), replace=False)
sampled_faraday_df = faraday_df.sample(n=min(sample_size, len(faraday_df)))
for customer_count in simulated_customer_count[:1]:  # Start with just one count for testing
    customer_count = 50
    indices, error = optimize_topology(
        customer_count, 
        feeder_ids, 
        sampled_timestamps, 
        grouped_horden, 
        sampled_faraday_df
    )
    if indices is not None:
        results.append({
            'customer_count': customer_count,
            'faraday_customer_indices_by_feeder': indices,
            'error': error
        })
    else:
        print(error)

In [32]:
def get_consumption_comparison(faraday_customer_indices_by_feeder):
    """
    Returns a DataFrame with timestamps and consumption values (both measured and simulated)
    for each feeder.
    
    Returns:
        pd.DataFrame with columns:
            - timestamp
            - feeder_id
            - measured_consumption
            - simulated_consumption
    """
    comparison_data = []
    
    for feeder_idx, feeder_id in enumerate(feeder_ids):
        feeder_customers = faraday_customer_indices_by_feeder[feeder_idx]
        selected_customers = faraday_df.iloc[feeder_customers]
        
        for timestamp in horden_timestamps:
            # Get measured consumption
            feeder_measurement = grouped_horden.get_group(
                (feeder_id, timestamp))['total_consumption_active_import'].iloc[0]
            
            # Calculate simulated consumption
            half_hour_idx = timestamp.hour * 2 + (timestamp.minute // 30)
            simulated_consumption = sum(
                float(eval(kwh)[half_hour_idx]) 
                for kwh in selected_customers['kwh']
            )
            
            comparison_data.append({
                'timestamp': timestamp,
                'feeder_id': feeder_id,
                'measured_consumption': feeder_measurement,
                'simulated_consumption': simulated_consumption
            })
    
    return pd.DataFrame(comparison_data)

# Example usage and plotting:
def plot_consumption_comparison(faraday_customer_indices_by_feeder):
    df = get_consumption_comparison(faraday_customer_indices_by_feeder)
    
    fig = go.Figure()
    
    for feeder_id in feeder_ids:
        feeder_data = df[df['feeder_id'] == feeder_id]
        
        # Add measured consumption line
        fig.add_trace(go.Scatter(
            x=feeder_data['timestamp'],
            y=feeder_data['measured_consumption'],
            name=f'{feeder_id} (Measured)',
            mode='lines',
            line=dict(dash='solid')
        ))
        
        # Add simulated consumption line
        fig.add_trace(go.Scatter(
            x=feeder_data['timestamp'],
            y=feeder_data['simulated_consumption'],
            name=f'{feeder_id} (Simulated)',
            mode='lines',
            line=dict(dash='dash')
        ))
    
    fig.update_layout(
        title='Measured vs Simulated Consumption by Feeder',
        xaxis_title='Timestamp',
        yaxis_title='Consumption',
        legend_title='Feeder'
    )
    
    fig.show()

# Find result with minimum error
min_error_idx = min(range(len(results)), key=lambda i: results[i]['error'])

print(f"Minimum error: {results[min_error_idx]['error']} ({math.sqrt(results[min_error_idx]['error']):.1f}^2 kWh)")
plot_consumption_comparison(results[min_error_idx]['faraday_customer_indices_by_feeder'])

Minimum error: 0.0 (0.0^2 kWh)
